# Service evidence
Run from the project root in a Databricks Git folder with the sibling `config/services.yaml`. Use compute with PySpark and PyYAML. The default input prefers the team's published clean table and falls back to the initial cleaning notebook's output.

In [0]:
%run ./00_config

In [0]:
from pathlib import Path
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window

config = yaml.safe_load(Path("config/services.yaml").read_text(encoding="utf-8"))
source = f"{OUT}.facilities_clean"
clean = spark.table(source)
clean.printSchema()
evidence_table = f"{OUT}.facility_service_evidence"
review_table = f"{OUT}.facility_service_review_v3"
print("Evidence source:", source, "| Services:", list(config["services"]))

In [0]:
import json
import re


def phrase_pattern(terms):
    # An empty list would compile to an empty alternation that matches between
    # punctuation marks, so it must never match instead.
    if not terms:
        return re.compile(r"(?!x)x")
    # Word boundaries avoid substring matches such as "MRI" in longer names.
    phrases = [r"\s+".join(re.escape(word) for word in t.split())
               for t in sorted(terms, key=len, reverse=True)]
    return re.compile(r"(?<!\w)(?:" + "|".join(phrases) + r")(?!\w)", re.I)


def compile_rules(config):
    rules = {}
    for service, terms in config["services"].items():
        rules[service] = {
            key: phrase_pattern(terms[key])
            for key in ("direct", "supporting", "specialties")
        }
        rules[service]["red_flags"] = phrase_pattern(config["red_flags"])
        rules[service]["service_veto"] = phrase_pattern(terms["red_flags"])
    return rules


def passages(value):
    # Preserve each quoted substring exactly, including capitalization.
    if isinstance(value, list):
        for item in value:
            yield from passages(item)
    elif isinstance(value, str):
        value = value.strip()
        if value.startswith("["):
            try:
                items = json.loads(value)
            except ValueError:
                return
            if isinstance(items, list):
                yield from passages(items)
            return
        for line in re.split(r"(?<=[.!?])\s+|[\r\n]+", value):
            if line.strip():
                yield line.strip()


def score_record(source_records, rules):
    evidence = []
    snippets = set()
    for record in source_records:
        for field, value in json.loads(record).items():
            snippets.update((field, line) for line in passages(value))

    for service, rule in rules.items():
        hits = []
        blocked = False
        for field, line in sorted(snippets):
            # Remove equipment phrases before testing direct terms. A machine name
            # alone does not document that the facility performs the procedure.
            remainder, equipment_hits = rule["supporting"].subn(" ", line)
            direct = bool(rule["direct"].search(remainder))
            specialty = bool(rule["specialties"].search(line))
            if rule["service_veto"].search(line):
                blocked = True
                continue
            if rule["red_flags"].search(line):
                # A service-specific contradiction vetoes even a positive duplicate.
                if direct or equipment_hits:
                    blocked = True
                continue
            if direct and field in {"procedure_list", "capability_list", "description"}:
                kind = "direct"
            elif equipment_hits or (field == "equipment_list" and direct):
                kind = "supporting"
            elif specialty:
                kind = "specialty"
            else:
                continue
            hits.append((kind, field, line))
        if blocked or not hits:
            continue
        direct_count = sum(kind == "direct" for kind, _, _ in hits)
        support_count = sum(kind == "supporting" for kind, _, _ in hits)
        if direct_count:
            level, score = "strong", 0.90 if support_count else 0.85
        elif support_count:
            level, score = "moderate", 0.55
        else:
            level, score = "weak", 0.20
        priority = {"direct": 0, "supporting": 1, "specialty": 2}
        hits.sort(key=lambda h: (priority[h[0]], h[1], h[2]))
        quotes = [(field, line) for _, field, line in hits]
        evidence.append((service, level, score, quotes, direct_count, support_count))
    return evidence

In [0]:
source = "workspace.datathon.facilities_clean"
clean = spark.table(source)
clean.printSchema()

### Input and scoring
Rows sharing an ID contribute to one evidence record. Each distinct (field, sentence) counts once, with direct evidence taking priority within a sentence. Equipment alone is moderate, a specialty alone is weak, and an explicit procedure or capability is strong. Scores are rule weights, not probabilities. Red flags in a service mention suppress that pair, even if another row is positive. Missing evidence and suppressed pairs are omitted from ranking. Specialty-only matches may apply to multiple services.

In [0]:
id_col = "unique_id"
fields = ["procedure_list", "equipment_list", "capability_list", "description", "specialties_list"]
# Keep the original ID for downstream joins and skip placeholder IDs.
ids = F.col(id_col).cast("string")
valid_id = ids.isNotNull() & ~F.lower(F.trim(ids)).isin("", "null", "none", "n/a", "na")
documents = (clean.where(valid_id)
             .select(ids.alias("unique_id"), F.to_json(F.struct(*fields)).alias("record"))
             .groupBy("unique_id").agg(F.sort_array(F.collect_set("record")).alias("source_records")))

rules = compile_rules(config)
schema = """array<struct<service:string,evidence_level:string,evidence_score:double,
            quotes:array<struct<field:string,sentence:string>>,n_direct:int,n_supporting:int>>"""
score_udf = F.udf(lambda records: score_record(records, rules), schema)
evidence = (documents.select("unique_id", F.explode(score_udf("source_records")).alias("hit"))
            .select("unique_id", "hit.*"))
evidence.write.format("delta").mode("overwrite").saveAsTable(evidence_table)
published = spark.table(evidence_table)
display(published.groupBy("service", "evidence_level").count().orderBy("service", "evidence_level"))
display(published.limit(10))

### Human review of 50 facility–service pairs
Freeze the sample once. Sample across strong, moderate, weak and unmatched pairs, taking approximately equal numbers per available tier. The result measures agreement on this audit sample. It is not a population-weighted accuracy estimate. Unmatched pairs make missed evidence observable. The review table snapshots source text and predictions together and retains labels on reruns. Use a new review table name for a new rule version.

In [0]:
if not spark.catalog.tableExists(review_table):
    service_ids = spark.createDataFrame([(s,) for s in rules], "service string")
    pairs = (documents.select("unique_id").crossJoin(service_ids)
             .join(published.select("unique_id", "service", "evidence_level"),
                   ["unique_id", "service"], "left")
             .fillna({"evidence_level": "none"}))
    order = [F.xxhash64("unique_id", "service", F.lit(2026)), "unique_id", "service"]
    strata = Window.partitionBy("evidence_level").orderBy(*order)
    sample = (pairs.withColumn("pos", F.row_number().over(strata))
              .orderBy("pos", *order).limit(50).drop("pos")
              .withColumnRenamed("evidence_level", "prediction")
              .join(documents, "unique_id")
              .withColumn("manual_level", F.lit(None).cast("string")))
    sample.write.format("delta").mode("ignore").saveAsTable(review_table)
review = spark.table(review_table)
print("Frozen sample size:", review.count())
display(review.select("unique_id", "service", "source_records", "manual_level")
        .orderBy("unique_id", "service"))

Read all original source fields for each pair without looking at the prediction. Assign **strong** for explicit on-site service evidence, **moderate** for supporting equipment, **weak** for a relevant specialty alone, and **none** when there is no credible evidence or explicit unavailability. Use human judgment for wording, negation scope and conflicts. Record the independent judgment even when the rule may disagree.

Use a SQL cell to set each label in the frozen table, replacing the ID, service and level in this example.

```sql
UPDATE workspace.datathon.facility_service_review_v3
SET manual_level = 'moderate'
WHERE unique_id = '<actual ID>' AND service = 'dialysis'
```

After completing all 50, rerun the next cell. It calculates exact four-level agreement and shows the confusion matrix.

In [0]:
review = spark.table(review_table)
valid_levels = ["strong", "moderate", "weak", "none"]
labelled = review.where(F.col("manual_level").isin(valid_levels))
n = review.count()
n_labelled = labelled.count()
if n == 50 and n_labelled == 50:
    correct = labelled.where(F.col("manual_level") == F.col("prediction")).count()
    print(f"Exact evidence-level accuracy on the audit sample: {correct}/50 = {correct / 50:.1%}")
    display(labelled.groupBy("manual_level").pivot("prediction", valid_levels)
            .count().fillna(0).orderBy("manual_level"))
else:
    print(f"{n_labelled} valid labels across {n} sampled pairs. Accuracy requires 50 reviewed pairs.")

In [0]:
print("当前读取的表：", source)
print("当前表的列名：", clean.columns)
display(clean.limit(5))

In [0]:
current_docs = documents.select(
    "unique_id", F.col("source_records").alias("current_records")
)
current_scores = published.select(
    "unique_id", "service", F.col("evidence_level").alias("current_level")
)

check = (
    spark.table(review_table)
    .join(current_docs, "unique_id", "left")
    .join(current_scores, ["unique_id", "service"], "left")
    .withColumn("current_level", F.coalesce("current_level", F.lit("none")))
)

display(check.select(
    F.count("*").alias("sample_size"),
    F.count_if(~F.col("source_records").eqNullSafe(F.col("current_records")))
     .alias("source_mismatches"),
    F.count_if(~F.col("prediction").eqNullSafe(F.col("current_level")))
     .alias("prediction_mismatches")
))

In [0]:
import json

sample = (
    spark.table(review_table)
    .where("manual_level IS NULL")
    .select("unique_id", "service", "source_records")
    .orderBy("unique_id", "service")
    .first()
)

if sample:
    print("ID:", sample.unique_id)
    print("Service to review:", sample.service)
    for record in sample.source_records:
        print(json.dumps(json.loads(record), indent=2, ensure_ascii=False))
else:
    print("All samples have been labeled.")